# Homework – Data Retrieval and Processing
## Smart Logistics Tracking System
**MO-IT148 | Blockchain and IoT Integration**

This notebook retrieves all stored IoT records from the blockchain,
merges with the original CSV to restore status and location fields,
and saves `cleaned_iot_data.csv`.

---
### Step 1 – Connect to Ganache and Load Contract

In [1]:
from web3 import Web3
import pandas as pd
import numpy as np

# Connect to local blockchain
ganache_url = 'http://127.0.0.1:7545'
web3 = Web3(Web3.HTTPProvider(ganache_url))

if web3.is_connected():
    print('✅ Connected to Ganache successfully!')
else:
    print('❌ Connection failed. Ensure Ganache is running.')

# ── REPLACE WITH YOUR CONTRACT ADDRESS ───────────────────────────────
contract_address = '0x769776d52E0987DAeF5B97D91CbA226A105e4c18'

abi = [
    {
        'inputs': [],
        'name': 'getTotalRecords',
        'outputs': [{'internalType': 'uint256', 'name': '', 'type': 'uint256'}],
        'stateMutability': 'view',
        'type': 'function'
    },
    {
        'inputs': [{'internalType': 'uint256', 'name': 'index', 'type': 'uint256'}],
        'name': 'getRecord',
        'outputs': [
            {'internalType': 'uint256', 'name': '', 'type': 'uint256'},
            {'internalType': 'uint256', 'name': '', 'type': 'uint256'}, 
            {'internalType': 'string',  'name': '', 'type': 'string'},
            {'internalType': 'string',  'name': '', 'type': 'string'},
            {'internalType': 'string',  'name': '', 'type': 'string'}
        ],
        'stateMutability': 'view',
        'type': 'function'
    },
    {
        'inputs': [
            {'internalType': 'uint256', 'name': '_dataTimestamp', 'type': 'uint256'},
            {'internalType': 'string',  'name': '_deviceId',  'type': 'string'},
            {'internalType': 'string',  'name': '_dataType',  'type': 'string'},
            {'internalType': 'string',  'name': '_dataValue', 'type': 'string'}
        ],
        'name': 'storeData',
        'outputs': [],
        'stateMutability': 'nonpayable',
        'type': 'function'
    }
]
# ─────────────────────────────────────────────────────────────────────

contract_address = Web3.to_checksum_address(contract_address)
contract = web3.eth.contract(address=contract_address, abi=abi)
print(f'✅ Connected to Smart Contract at {contract_address}')

✅ Connected to Ganache successfully!
✅ Connected to Smart Contract at 0x769776d52E0987DAeF5B97D91CbA226A105e4c18


---
### Step 2 – Retrieve All IoT Records from Blockchain

In [2]:
# Get total stored records
total_records = contract.functions.getTotalRecords().call()
print(f'Total IoT records stored: {total_records}')

data = []
for i in range(total_records):
    record = contract.functions.getRecord(i).call()
    data.append({
        'timestamp':       record[0], 
        'block_timestamp': record[1],   
        'device_id':       record[2],
        'data_type':       record[3],
        'data_value':      record[4]
    })

df_chain = pd.DataFrame(data)
df_chain['timestamp']       = pd.to_datetime(df_chain['timestamp'], unit='s')
df_chain['block_timestamp'] = pd.to_datetime(df_chain['block_timestamp'], unit='s')

print(f'\n✅ Retrieved {len(df_chain)} records from blockchain')
print(df_chain.head())

Total IoT records stored: 100

✅ Retrieved 100 records from blockchain
            timestamp     block_timestamp device_id    data_type  \
0 2026-01-01 01:30:10 2026-07-05 16:37:00    RFID_0          GPS   
1 2026-01-01 02:30:22 2026-07-05 16:37:01    RFID_1          GPS   
2 2026-01-01 03:28:12 2026-07-05 16:37:02    RFID_2          GPS   
3 2026-01-01 04:30:02 2026-07-05 16:37:03    RFID_3  Temperature   
4 2026-01-01 10:44:47 2026-07-05 16:37:04    RFID_4  Temperature   

             data_value  
0  14.949141,124.334215  
1  10.528973,124.910757  
2  11.412766,124.552706  
3                 5.15C  
4                 7.31C  


---
### Step 3 – Merge with Original CSV to Restore Status and Location

In [3]:
# Load original IoT dataset (contains status, latitude, longitude)
df_original = pd.read_csv('logistics_iot_data.csv')

df_merged = df_chain.merge(
    df_original[['device_id', 'package_id', 'status', 'latitude', 'longitude']],
    on='device_id',
    how='left'
)

print('✅ Merged blockchain data with original CSV')
print(f'   Columns now available: {list(df_merged.columns)}')
print(df_merged.head())

✅ Merged blockchain data with original CSV
   Columns now available: ['timestamp', 'block_timestamp', 'device_id', 'data_type', 'data_value', 'package_id', 'status', 'latitude', 'longitude']
            timestamp     block_timestamp device_id    data_type  \
0 2026-01-01 01:30:10 2026-07-05 16:37:00    RFID_0          GPS   
1 2026-01-01 02:30:22 2026-07-05 16:37:01    RFID_1          GPS   
2 2026-01-01 03:28:12 2026-07-05 16:37:02    RFID_2          GPS   
3 2026-01-01 04:30:02 2026-07-05 16:37:03    RFID_3  Temperature   
4 2026-01-01 10:44:47 2026-07-05 16:37:04    RFID_4  Temperature   

             data_value  package_id      status   latitude   longitude  
0  14.949141,124.334215        1055     Delayed  14.949141  124.334215  
1  10.528973,124.910757        1066   Delivered  10.528973  124.910757  
2  11.412766,124.552706        1001  In Transit  11.412766  124.552706  
3                 5.15C        1081     Delayed  10.614545  123.218513  
4                 7.31C        1086

---
### Step 4 – Process and Clean Data

In [4]:
# Extract numeric values from data_value column
# Temperature: '6.24C'         → 6.24
# Humidity:    '72.5%'         → 72.5
# GPS:         '12.42,121.19'  → 12.42 (first coordinate)
df_merged['numeric_value'] = df_merged['data_value'].str.extract(r'(\d+\.?\d*)').astype(float)

# Handle any missing values
df_merged.fillna(0, inplace=True)

print('✅ Data cleaned successfully!')
print(f'   Sensor type breakdown : {df_merged["data_type"].value_counts().to_dict()}')
print(f'   Status breakdown      : {df_merged["status"].value_counts().to_dict()}')
print()
print('Cleaned data preview:')
print(df_merged[['timestamp','device_id','data_type','data_value',
                  'numeric_value','status','latitude','longitude']].head())

✅ Data cleaned successfully!
   Sensor type breakdown : {'Temperature': 34, 'GPS': 33, 'Humidity': 33}
   Status breakdown      : {'Delayed': 44, 'In Transit': 34, 'Delivered': 22}

Cleaned data preview:
            timestamp device_id    data_type            data_value  \
0 2026-01-01 01:30:10    RFID_0          GPS  14.949141,124.334215   
1 2026-01-01 02:30:22    RFID_1          GPS  10.528973,124.910757   
2 2026-01-01 03:28:12    RFID_2          GPS  11.412766,124.552706   
3 2026-01-01 04:30:02    RFID_3  Temperature                 5.15C   
4 2026-01-01 10:44:47    RFID_4  Temperature                 7.31C   

   numeric_value      status   latitude   longitude  
0      14.949141     Delayed  14.949141  124.334215  
1      10.528973   Delivered  10.528973  124.910757  
2      11.412766  In Transit  11.412766  124.552706  
3       5.150000     Delayed  10.614545  123.218513  
4       7.310000   Delivered  12.170094  121.315296  


---
### Step 5 – Save Cleaned Data

In [5]:
# Final column
df_clean = df_merged[[
    'timestamp',
    'device_id',
    'package_id',
    'data_type',       
    'data_value',
    'numeric_value',   
    'status',          
    'latitude',
    'longitude'
]].copy()

# Save cleaned IoT data to CSV
df_clean.to_csv('cleaned_iot_data.csv', index=False)

print('✅ Cleaned IoT data saved successfully as cleaned_iot_data.csv')
print(f'   Columns    : {list(df_clean.columns)}')
print(f'   Shape      : {df_clean.shape[0]} rows x {df_clean.shape[1]} columns')

✅ Cleaned IoT data saved successfully as cleaned_iot_data.csv
   Columns    : ['timestamp', 'device_id', 'package_id', 'data_type', 'data_value', 'numeric_value', 'status', 'latitude', 'longitude']
   Shape      : 100 rows x 9 columns
